In [50]:
import pandas as pd
import itertools
from collections import Counter
import numpy as np

df = pd.read_csv('/root/data/mimic-iv_data/hosp/diagnoses_icd.csv')


# Add ICD version to code
df['icd_code'] = df['icd_code'] + '-' + df['icd_version'].astype(str)

In [2]:
df

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723-9,9
1,10000032,22595853,2,78959-9,9
2,10000032,22595853,3,5715-9,9
3,10000032,22595853,4,07070-9,9
4,10000032,22595853,5,496-9,9
...,...,...,...,...,...
5006879,19999987,23865745,7,41401-9,9
5006880,19999987,23865745,8,78039-9,9
5006881,19999987,23865745,9,0413-9,9
5006882,19999987,23865745,10,36846-9,9


In [3]:
unique_codes = sorted(set(df['icd_code']))
len(unique_codes)

26164

In [4]:
# Checking special token IDs for BERT

# from transformers import BertTokenizer
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# text = "[CLS][PAD] [UNK] [SEP] This is a test sentence. and"
# encoding = tokenizer.encode_plus(text, add_special_tokens = True,    truncation = True, return_tensors = "pt")
# print(encoding)

In [52]:
# Use BERT tokenizer settings

# Get special tokens
og_bert_token_path = "./data/og_bert_tokens.txt"
special_tokens = []
with open(og_bert_token_path, 'r') as f:
     lines = f.readlines()
     for line in lines:
        special_tokens.append(line.strip())

vocab = {str(code): i for i, code in enumerate(unique_codes, start=len(special_tokens))}

# Add special tokens to vocab
for i in np.arange(len(special_tokens)):
    vocab[special_tokens[i]] = str(i)


import json
with open('./data/vocab.json', 'w') as f:
    json.dump(vocab, f)

In [53]:
with open('./data/icd_9_10_vocab.txt', 'w') as f:
    f.write('\n'.join(unique_codes))

In [57]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

from tokenizers.trainers import WordLevelTrainer
trainer = WordLevelTrainer(special_tokens=special_tokens)

In [58]:
tokenizer.train(['./data/icd_9_10_vocab.txt'], trainer)

In [59]:
tokenizer.token_to_id('[PAD]'), tokenizer.token_to_id('[SEP]'), tokenizer.token_to_id('[CLS]'),  tokenizer.token_to_id('[UNK]'), tokenizer.token_to_id('5')

(0, 102, 101, 100, None)

In [10]:
from tokenizers.processors import BertProcessing
tokenizer.post_processor = BertProcessing(
    sep=('[SEP]', tokenizer.token_to_id('[SEP]')), 
    cls=('[CLS]', tokenizer.token_to_id('[CLS]')))

In [45]:
tokenizer.get_vocab()

{'5368': 3667,
 'A390': 7831,
 'I973': 13331,
 'S42341A': 20201,
 '1953': 753,
 '25070': 1254,
 'G8381': 11423,
 'G2579': 11076,
 'C140': 8234,
 'V4582': 24175,
 'V51': 24266,
 'Z85840': 25874,
 'R188': 18243,
 'S82391G': 21662,
 'M8888': 16466,
 'S32411A': 19785,
 'C138': 8232,
 '43401': 3037,
 'Y92211': 25195,
 '9651': 7464,
 'O328XX0': 17337,
 'Y35391A': 25030,
 '82525': 6534,
 'A7489': 7931,
 'C577': 8495,
 'J9612': 13627,
 'A033': 7764,
 'I63139': 12672,
 'M71121': 16024,
 '1912': 731,
 'H4031X0': 11935,
 'I69120': 12802,
 'N491': 16786,
 '74739': 5626,
 'T5991XA': 23032,
 'S73001A': 21389,
 '28521': 1521,
 'I63533': 12727,
 'V1988XA': 23993,
 'V271': 24051,
 'Z62898': 25646,
 'L97519': 14775,
 '59683': 4007,
 'S82391B': 21660,
 '7947': 6084,
 '7593': 5797,
 'R8271': 18629,
 'V625': 24418,
 'H5988': 12160,
 'S63602A': 21025,
 'S01512D': 18777,
 '30651': 1870,
 'S5002XA': 20355,
 '80238': 6244,
 'R498': 18525,
 '78931': 6008,
 'M62152': 15870,
 '1900': 725,
 'G5770': 11317,
 '71954

In [11]:
tokenizer.save('./data/tokenizer-icd.json')

In [12]:
text = ' '.join(unique_codes[:200])
output = tokenizer.encode(text,)

In [13]:
from transformers import PreTrainedTokenizerFast

t = PreTrainedTokenizerFast(tokenizer_file='./data/tokenizer-icd.json')
t.add_special_tokens({'pad_token': '[PAD]', 'mask_token': '[MASK]'})

0

In [14]:
t.save_pretrained('./data/tokenizer-icd-final')

('./data/tokenizer-icd-final/tokenizer_config.json',
 './data/tokenizer-icd-final/special_tokens_map.json',
 './data/tokenizer-icd-final/tokenizer.json')

In [15]:
encoding = t(
            text,
            return_offsets_mapping=False,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_overflowing_tokens=False)

In [16]:
t(unique_codes[:12], is_split_into_words=True, padding='max_length', max_length=128)

{'input_ids': [1, 29, 5, 7, 30, 5, 7, 31, 5, 7, 32, 5, 7, 33, 5, 7, 34, 5, 7, 35, 5, 7, 36, 5, 7, 37, 5, 7, 38, 5, 7, 39, 5, 7, 40, 5, 7, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 